### Wikipedia Retriever

In [4]:
from langchain_community.retrievers import WikipediaRetriever

In [5]:
# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang='en')

In [6]:
# Define your query
query = "the geopolitical history of India and pakistan from the perspective of a chinese"

#get relevant wikipedia documents
docs = retriever.invoke(query)

In [7]:
#print retrieved content
for index,document in enumerate(docs,start=1):
    print(f'\n--Result{index}--')
    print(f'Content:\n{document.page_content}...')



--Result1--
Content:
China and India maintained peaceful relations for thousands of years, but their relationship has varied since the Chinese Communist Party (CCP)'s victory in the Chinese Civil War in 1949 and the annexation of Tibet by the People's Republic of China. The two nations have sought economic cooperation with each other, while frequent border disputes and economic nationalism in both countries are major points of contention.
Cultural and economic relations between China and India date back to ancient times. The Silk Road not only served as a major trade route between India and China, but is also credited for facilitating the spread of Buddhism from India to East Asia. During the 19th century, China was involved in a growing opium trade with the East India Company, which exported opium grown in India. During World War II, both British India and the Republic of China (ROC) played a crucial role in halting the progress of Imperial Japan. After India became independent in 19

### Vector Store Retriever

In [8]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEndpointEmbeddings, HuggingFaceEmbeddings
from langchain.schema import Document
from dotenv import load_dotenv

In [9]:
load_dotenv()

embedding_model = HuggingFaceEndpointEmbeddings(
    repo_id="sentence-transformers/all-MiniLM-L6-v2",
    task="feature-extraction"
)



In [10]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [11]:
#Create Chroma vector store in memeory

vector_store = Chroma.from_documents(
    documents=documents,
    embedding= embedding_model,
    collection_name='my_collection'
)

In [12]:
# Step 4: Convert vectorstore into a retriever
retriever = vector_store.as_retriever(search_kwargs={'k':2})

In [13]:
query = "What is Chroma used for?"
response = retriever.invoke(query)

In [14]:
for i, doc in enumerate(response,start=1):
    print(f'\nResult {i}:')
    print(f'{doc.page_content} ')


Result 1:
Chroma is a vector database optimized for LLM-based search. 

Result 2:
LangChain helps developers build LLM applications easily. 


### MMR

In [15]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [16]:
from langchain_community.vectorstores import FAISS

In [17]:
# Step 2: Create the FAISS/or use Chroma vector store from documents
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=docs,
    embedding= embedding_model
)

In [18]:
#Enable MMR in the retriever
retriever = vector_store.as_retriever(
    search_type = "mmr", # <-- This enables MMR
    search_kwargs = {"k":2, "lambda_mult":0.5}   # k = top results, lambda_mult = relevance-diversity balance
)

In [19]:
query = "What is langchain?"
result = retriever.invoke(query)

In [20]:
for i, doc in enumerate(result,start=1):
    print(f'\nResult {i}:')
    print(doc.page_content)


Result 1:
LangChain supports Chroma, FAISS, Pinecone, and more.

Result 2:
LangChain is used to build LLM based applications.


### Multi Query Retriever

In [21]:
from langchain.retrievers.multi_query import MultiQueryRetriever

In [22]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [23]:
#create Chroma vector store

vector_store = Chroma.from_documents(
    documents=all_docs,
    embedding=embedding_model
)

In [24]:
# Create retrievers
similarity_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [25]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation",
    temperature= 1.5
    
)


model = ChatHuggingFace(llm = llm)

In [26]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={'k':2}),
    llm = model
)

In [27]:
#query
query = "How to improve energy levels and maintain balance?"

In [28]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [29]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 5 ---
Photosynthesis enables plants to produce energy by converting sunlight.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.


### Contextual Compression Retriever

In [30]:
from langchain_chroma import Chroma
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [31]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [32]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation",
    temperature= 1.5
    
)


model = ChatHuggingFace(llm = llm)

In [33]:
vector_store = Chroma.from_documents(documents=docs,embedding=embedding_model)

In [34]:
base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [35]:
# Set up the compressor using an LLM
llm = model
compressor = LLMChainExtractor.from_llm(llm)

In [36]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [38]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

HfHubHTTPError: 402 Client Error: Payment Required for url: https://router.huggingface.co/together/v1/chat/completions (Request ID: Root=1-68bfd80a-63222639193bad1f490759e2;b1722d44-2f44-41a8-a3f9-fd7cf4ceeec8)

You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.

In [ ]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)
